In [ ]:
from tqdm import tqdm
import numpy as np

import sympy as sp
from sympy import *

In [ ]:
x = symbols('x', real=True)
lam = symbols('lambda', real=True, positive=True)

## Trigonometric Case

In [ ]:
def get_trig_eigenstate_raising(n):
    aux_norm = sqrt(gamma(lam + n + 1) / (sqrt(pi) * gamma(lam + n + Rational(1, 2))))
    psi_0 = aux_norm * cos(x)**(lam + n)
    norm = sqrt(gamma(2 * lam + n) / (gamma(n+1) * gamma(2 * lam + 2 * n)))
    result = psi_0
    for k in range(n, 0, -1): # raise psi_0 successively
        result = -I * diff(result, x) + I * (lam + k) * tan(x) * result
    return norm * result

In [ ]:
def get_trig_eigenstate_hyper(n):
    a = -n
    b = n + 2*lam
    c = lam + Rational(1, 2)
    z = (1 + sin(x)) / 2

    psi = (cos(x)) ** lam * hyper([a, b], [c], z)
    norm = sqrt( (2*(n + lam) * gamma(n + 2*lam))
                 / ( factorial(n) * 2**(2*lam) * gamma(lam + Rational(1,2))**2 ) )
    return (norm * psi)

In [ ]:
trig_wavefunctions = []
for n in tqdm(range(100)):
    trig_wavefunctions.append(get_trig_eigenstate_raising(n))

In [ ]:
Integral(abs(trig_wavefunctions[-1].subs(lam, 3))**2, (x, -pi/2, pi/2)).evalf()

In [ ]:
def change_basis_trigonometric(psi_0, lam_val, max_n=10, precision=4):
    basis = []
    domain = np.linspace(-np.pi/2, np.pi/2, 10**precision)
    psi_0 = lambdify(x, psi_0.subs(lam, lam_val), 'numpy')
    for n in tqdm(range(max_n + 1)):
        energy, psi_n = get_eigenstate_trigonometric(n)
        energy = float(energy.evalf(subs={lam: lam_val}))
        psi_n = lambdify(x, conjugate(psi_n.subs(lam, lam_val)), 'numpy')
        coeff = np.trapz(psi_n(domain) * psi_0(domain), x=domain)
        basis.append((energy, coeff, psi_n))
    
    return basis

def time_evolve_trigonometric(psi_transformed, max_t, frames=1000, tol=0.01):
    domain = np.linspace(-np.pi/2, np.pi/2, 1000)
    def psi_t(t_val):
        psi_t_vals = np.zeros_like(domain, dtype=complex)
        for energy, coeff, psi_n in psi_transformed:
            psi_t_vals += coeff * psi_n(domain) * np.exp(-1j * energy * t_val)
        return psi_t_vals
    
    psi_evolution = []
    for t in tqdm(np.linspace(0, max_t, frames)):
        psi_t_val = psi_t(t)
        if psi_evolution and abs(psi_t_val - psi_evolution[0][1]).max() < tol:
            print(f"Full period. Stopping evolution.")
            break
        psi_evolution.append((t, psi_t_val))
    return psi_evolution

In [ ]:
# psi_0 = (get_eigenstate_trigonometric(0)[1] + get_eigenstate_trigonometric(1)[1] + get_eigenstate_trigonometric(5)[1]) / sqrt(3)
psi_0 = simplify(exp(-x**2)*cos(x))

In [ ]:
transformed = change_basis_trigonometric(psi_0, lam_val=3, max_n=5)
evolution = time_evolve_trigonometric(transformed, max_t=1, frames=100)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
from tqdm import tqdm

frames = []

# x values from evolution data
x_vals = np.linspace(-np.pi/2, np.pi/2, len(evolution[0][1]))
real_min, real_max = np.inf, -np.inf
imag_min, imag_max = np.inf, -np.inf

for _, psi_t in evolution:
    real_min = min(real_min, psi_t.real.min())
    real_max = max(real_max, psi_t.real.max())
    imag_min = min(imag_min, psi_t.imag.min())
    imag_max = max(imag_max, psi_t.imag.max())

x_min, x_max = x_vals.min(), x_vals.max()

N = len(evolution)
for k, (t, psi_t) in enumerate(tqdm(evolution)):

    fig = plt.figure(figsize=(7,5))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot(
        x_vals,
        psi_t.imag,
        psi_t.real,
        linewidth=2
    )

    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("Im ψ(x,t)", fontsize=12)
    ax.set_zlabel("Re ψ(x,t)", fontsize=12)
    ax.set_title(f"Wavefunction at t = {t:.2f}")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(imag_min, imag_max)
    ax.set_zlim(real_min, real_max)

    # rotate
    ax.view_init(elev=30, azim=360 * k/N)

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=120)
    plt.close(fig)
    buf.seek(0)
    frames.append(Image.open(buf).convert("RGB"))

frames[0].save(
    "psi_complex_curve.gif",
    save_all=True,
    append_images=frames[1:],
    duration=80,
    loop=0
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
from tqdm import tqdm

frames = []

# x values from evolution data
x_vals = np.linspace(-np.pi/2, np.pi/2, len(evolution[0][1]))

# compute global min/max for |psi|^2
prob_min, prob_max = np.inf, -np.inf
for _, psi_t in evolution:
    prob = np.abs(psi_t)**2
    prob_min = min(prob_min, prob.min())
    prob_max = max(prob_max, prob.max())

x_min, x_max = x_vals.min(), x_vals.max()

N = len(evolution)

for k, (t, psi_t) in enumerate(tqdm(evolution)):
    
    fig = plt.figure(figsize=(7,5))
    ax = fig.add_subplot(111)

    # probability density
    prob = np.abs(psi_t)**2

    ax.plot(x_vals, prob, linewidth=2)

    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("|ψ(x,t)|²", fontsize=12)
    ax.set_title(f"Probability Density at t = {t:.2f}")

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(prob_min, prob_max)

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=120)
    plt.close(fig)
    buf.seek(0)
    frames.append(Image.open(buf).convert("RGB"))

frames[0].save(
    "psi_prob_density.gif",
    save_all=True,
    append_images=frames[1:],
    duration=80,
    loop=0
)
